In [1]:
import gc
import json
import math
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import skew
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

BACKEND_DIR = Path("..").resolve()
KB_DIR = BACKEND_DIR / "knowledge_base"
KB_JSON = KB_DIR / "knowledge_base.json"
UPITI_JSON = KB_DIR / "tekst_upiti.json"
VAN_TEME_JSON = KB_DIR / "tekst_upiti_van_teme.json"
LAICKI_JSON = KB_DIR / "tekst_upiti_ai.json"
POREDJENJE_CSV = BACKEND_DIR / "eval_text_modeli.csv"

MODELI = ["BAAI/bge-small-en-v1.5", "BAAI/bge-base-en-v1.5", "BAAI/bge-large-en-v1.5"]
PREFIKS_UPITA = "Represent this sentence for searching relevant passages: "
BATCH_SIZE = 16
TEZINA_DOKUMENTA = 0.3
MAX_TOKENA = 500
SEED = 42
B_BOOTSTRAP = 2000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE =", DEVICE)

DEVICE = cuda


In [2]:
with open(KB_JSON, encoding="utf-8") as f:
    knowledge_base = json.load(f)
with open(UPITI_JSON, encoding="utf-8") as f:
    svi_upiti = json.load(f)
with open(VAN_TEME_JSON, encoding="utf-8") as f:
    van_teme = json.load(f)["upiti"]
with open(LAICKI_JSON, encoding="utf-8") as f:
    laicki = json.load(f)["upiti"]

SIDS = sorted(knowledge_base)
BROJ_VRSTA = len(SIDS)
INDEKS = {}
for j, sid in enumerate(SIDS):
    INDEKS[sid] = j

redovi = []
for sid in svi_upiti:
    if sid == "_meta":
        continue
    for stavka in svi_upiti[sid]["val"]:
        redovi.append({"specie_id": sid, "stil": stavka["stil"], "upit": stavka["upit"]})
val_df = pd.DataFrame(redovi)

TACNA_VAL = np.array([INDEKS[sid] for sid in val_df["specie_id"]])
TACNA_LAICKI = np.array([INDEKS[sid] for sid in laicki])

print("Vrsta:", BROJ_VRSTA, "| val upita:", len(val_df), "| laičkih:", len(laicki), "| van teme:", len(van_teme))

Vrsta: 324 | val upita: 648 | laičkih: 18 | van teme: 40


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODELI[0])

def broj_tokena(tekst):
    return len(tokenizer.encode(tekst, add_special_tokens=False))


def ocisti_naslove(tekst):
    tekst = re.sub(r"\s*^=+[^\n]*=+\s*\Z", "", tekst, flags=re.MULTILINE)
    tekst = re.sub(r"^=+\s*(.+?)\s*=+[ \t]*\n\s*", r"\1: ", tekst, flags=re.MULTILINE)
    return tekst.strip()


KRAJ_RECENICE = r"(?<=[.!?])\s+(?=[A-Z])"

SEPARATORI = ["\n\n\n", "\n\n", "\n", KRAJ_RECENICE]


def podeli(tekst, separator):
    if separator == KRAJ_RECENICE:
        delovi = re.split(KRAJ_RECENICE, tekst)
    else:
        delovi = tekst.split(separator)
    rezultat = []
    for deo in delovi:
        if deo.strip() != "":
            rezultat.append(deo.strip())
    return rezultat


def spoji_u_chunkove(celine, spojnica):
    ukupno = 0
    for celina in celine:
        ukupno += broj_tokena(celina)
    broj_delova = math.ceil(ukupno / MAX_TOKENA)
    cilj = ukupno / broj_delova

    chunkovi = []
    tekuci = []
    tekuci_tokeni = 0
    for celina in celine:
        n = broj_tokena(celina)
        if len(tekuci) > 0:
            novo = tekuci_tokeni + n
            if novo > MAX_TOKENA or abs(novo - cilj) > abs(tekuci_tokeni - cilj):
                chunkovi.append(spojnica.join(tekuci))
                tekuci = []
                tekuci_tokeni = 0
        tekuci.append(celina)
        tekuci_tokeni += n
    chunkovi.append(spojnica.join(tekuci))

    if len(chunkovi) > 1 and broj_tokena(chunkovi[-2]) + broj_tokena(chunkovi[-1]) <= MAX_TOKENA:
        poslednji = chunkovi.pop()
        chunkovi[-1] = chunkovi[-1] + spojnica + poslednji
    return chunkovi


def podeli_na_chunkove(tekst, nivo=0):
    if broj_tokena(tekst) <= MAX_TOKENA:
        return [tekst]

    separator = SEPARATORI[nivo]
    celine = podeli(tekst, separator)

    if len(celine) == 1:
        return podeli_na_chunkove(tekst, nivo + 1)

    delovi = []
    for celina in celine:
        if broj_tokena(celina) <= MAX_TOKENA:
            delovi.append(celina)
        else:
            delovi += podeli_na_chunkove(celina, nivo + 1)

    if separator == KRAJ_RECENICE:
        spojnica = " "
    else:
        spojnica = separator
    return spoji_u_chunkove(delovi, spojnica)


def recenice(tekst):
    lista = []
    for red in podeli(tekst, "\n"):
        lista += podeli(red, KRAJ_RECENICE)
    return lista


def prozori(tekst):
    r = recenice(tekst)
    if len(r) <= 2:
        return [" ".join(r)]
    lista = []
    for i in range(len(r) - 1):
        lista.append(r[i] + " " + r[i + 1])
    return lista


def napravi(nivo):
    tekstovi = []
    vlasnik = []
    for j, sid in enumerate(SIDS):
        tekst = ocisti_naslove(knowledge_base[sid]["tekst"])
        if nivo == "prozor":
            delovi = prozori(tekst)
        else:
            delovi = podeli_na_chunkove(tekst)
        tekstovi += delovi
        vlasnik += [j] * len(delovi)
    return tekstovi, np.array(vlasnik)


PROZORI, PROZORI_VLASNIK = napravi("prozor")
DOKUMENTI, DOKUMENTI_VLASNIK = napravi("dokument")
print("Prozora:", len(PROZORI), "| delova dokumenata:", len(DOKUMENTI))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1108 > 512). Running this sequence through the model will result in indexing errors


Prozora: 2509 | delova dokumenata: 349


In [4]:
def centriraj(v, mu):
    v = v - mu
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def skor_vrste(sim, vlasnik):
    kolone = []
    for j in range(BROJ_VRSTA):
        kolone.append(sim[:, vlasnik == j].max(axis=1))
    return np.column_stack(kolone)


def rang_tacne(S, tacna):
    rangovi = []
    for i in range(len(S)):
        poredak = list(np.argsort(-S[i]))
        rangovi.append(poredak.index(tacna[i]) + 1)
    return np.array(rangovi)


def mere(S, tacna):
    rang = rang_tacne(S, tacna)
    rezultat = {}
    rezultat["Hit@1"] = np.mean(rang <= 1) * 100
    rezultat["Hit@3"] = np.mean(rang <= 3) * 100
    rezultat["Hit@10"] = np.mean(rang <= 10) * 100
    rezultat["MRR"] = np.mean(1 / rang)

    sortirano = np.sort(S, axis=1)
    rezultat["gap_medijana"] = np.median(sortirano[:, -1] - sortirano[:, -2])

    prva = np.bincount(S.argmax(axis=1), minlength=BROJ_VRSTA)
    deset_posto = round(BROJ_VRSTA * 0.10)
    rezultat["10%_vrsta_uzima_%"] = np.sort(prva)[::-1][:deset_posto].sum() / len(S) * 100
    rezultat["nikad_prve"] = int((prva == 0).sum())

    for k in [1, 5, 10]:
        prvih_k = np.argsort(-S, axis=1)[:, :k]
        N_k = np.bincount(prvih_k.ravel(), minlength=BROJ_VRSTA)
        rezultat[f"skew_N{k}"] = skew(N_k)
        rezultat[f"antihub_N{k}"] = int((N_k == 0).sum())
    return rezultat


def embeduj(model, tekstovi):
    return model.encode(tekstovi, batch_size=BATCH_SIZE, normalize_embeddings=True)


def izmeri(ime_modela):
    model = SentenceTransformer(ime_modela, device=DEVICE)

    start = time.time()
    P = embeduj(model, PROZORI)
    D = embeduj(model, DOKUMENTI)
    vreme_baze = time.time() - start

    Q_val = embeduj(model, [PREFIKS_UPITA + u for u in val_df["upit"]])
    Q_laicki = embeduj(model, [PREFIKS_UPITA + u for u in laicki.values()])
    Q_van = embeduj(model, [PREFIKS_UPITA + u for u in van_teme])

    start = time.time()
    for u in list(laicki.values())[:10]:
        model.encode(PREFIKS_UPITA + u, normalize_embeddings=True)
    ms_po_upitu = (time.time() - start) / 10 * 1000

    mu_upit = Q_val.mean(axis=0)
    P_c = centriraj(P, P.mean(axis=0))
    D_c = centriraj(D, D.mean(axis=0))
    Q_val_c = centriraj(Q_val, mu_upit)
    Q_laicki_c = centriraj(Q_laicki, mu_upit)

    S_prozor = skor_vrste(Q_val_c @ P_c.T, PROZORI_VLASNIK)
    S_dokument = skor_vrste(Q_val_c @ D_c.T, DOKUMENTI_VLASNIK)
    S_val = (1 - TEZINA_DOKUMENTA) * S_prozor + TEZINA_DOKUMENTA * S_dokument
    S_laicki = ((1 - TEZINA_DOKUMENTA) * skor_vrste(Q_laicki_c @ P_c.T, PROZORI_VLASNIK)
                + TEZINA_DOKUMENTA * skor_vrste(Q_laicki_c @ D_c.T, DOKUMENTI_VLASNIK))

    sirov_val = (Q_val @ P.T).max(axis=1)
    sirov_van = (Q_van @ P.T).max(axis=1)
    oznake = [1] * len(sirov_val) + [0] * len(sirov_van)
    auc = roc_auc_score(oznake, np.concatenate([sirov_val, sirov_van]))

    if DEVICE == "cuda":
        gpu = torch.cuda.max_memory_allocated() / 1e9
    else:
        gpu = np.nan

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    return {
        "S_val": S_val,
        "S_prozor": S_prozor,
        "S_dokument": S_dokument,
        "rang_val": rang_tacne(S_val, TACNA_VAL),
        "rang_laicki": rang_tacne(S_laicki, TACNA_LAICKI),
        "auc_van_teme": auc,
        "vreme_baze_s": vreme_baze,
        "ms_po_upitu": ms_po_upitu,
        "gpu_GB": gpu,
    }


REZULTATI = {}
for ime in MODELI:
    REZULTATI[ime] = izmeri(ime)
    print(ime, "gotov")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3893.29it/s]

BAAI/bge-small-en-v1.5 gotov


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   6%|▌         | 11/199 [00:00<00:01, 99.35it/s]

Loading weights:  11%|█         | 21/199 [00:00<00:02, 63.29it/s]

... (12 outputs left out)

Loading weights:   8%|▊         | 32/391 [00:01<00:24, 14.40it/s]

Loading weights:   9%|▊         | 34/391 [00:01<00:29, 12.08it/s]

Loading weights: 100%|██████████| 391/391 [00:02<00:00, 193.33it/s]

BAAI/bge-large-en-v1.5 gotov


In [5]:
tabela = {}
for ime, r in REZULTATI.items():
    red = mere(r["S_val"], TACNA_VAL)
    for k in [1, 3, 10]:
        red[f"laicki Hit@{k}"] = np.mean(r["rang_laicki"] <= k) * 100
    red["AUC_van_teme"] = r["auc_van_teme"]
    red["baza_s"] = r["vreme_baze_s"]
    red["ms_po_upitu"] = r["ms_po_upitu"]
    red["GPU_GB"] = r["gpu_GB"]
    tabela[ime.split("/")[1]] = red

tabela = pd.DataFrame(tabela).T
tabela.to_csv(POREDJENJE_CSV, encoding="utf-8-sig")
tabela.round(3)

,Hit@1,Hit@3,Hit@10,MRR,gap_medijana,10%_vrsta_uzima_%,nikad_prve,skew_N1,antihub_N1,skew_N5,antihub_N5,skew_N10,antihub_N10,laicki Hit@1,laicki Hit@3,laicki Hit@10,AUC_van_teme,baza_s,ms_po_upitu,GPU_GB
bge-small-en-v1.5,43.827,63.272,77.315,0.557,0.037,31.327,69.0,1.933,69.0,1.632,4.0,1.486,0.0,22.222,38.889,66.667,1.0,14.374,22.440,0.292
bge-base-en-v1.5,49.537,65.432,83.179,0.605,0.040,31.481,56.0,2.568,56.0,1.806,4.0,1.644,1.0,22.222,44.444,61.111,1.0,65.015,30.848,0.742
bge-large-en-v1.5,55.247,72.377,85.648,0.659,0.050,29.475,49.0,2.014,49.0,1.817,2.0,1.422,0.0,33.333,38.889,83.333,1.0,153.552,54.483,1.741


In [6]:
rng = np.random.default_rng(SEED)
indeksi_po_vrsti = val_df.groupby("specie_id").indices
vrste = list(indeksi_po_vrsti)

uzorci = []
for b in range(B_BOOTSTRAP):
    uzorak_vrsta = rng.choice(vrste, len(vrste))
    uzorci.append(np.concatenate([indeksi_po_vrsti[v] for v in uzorak_vrsta]))

rang_small = REZULTATI[MODELI[0]]["rang_val"]
redovi = []
for ime in MODELI[1:]:
    rang = REZULTATI[ime]["rang_val"]
    for k in [1, 3]:
        razlike = []
        for idx in uzorci:
            razlike.append((np.mean(rang[idx] <= k) - np.mean(rang_small[idx] <= k)) * 100)
        redovi.append({
            "model": ime.split("/")[1],
            "mera": f"Hit@{k}",
            "razlika_pp": (np.mean(rang <= k) - np.mean(rang_small <= k)) * 100,
            "p2.5": np.percentile(razlike, 2.5),
            "p97.5": np.percentile(razlike, 97.5),
        })

razlike_df = pd.DataFrame(redovi)
razlike_df.round(2)

,model,mera,razlika_pp,p2.5,p97.5
0,bge-base-en-v1.5,Hit@1,5.71,2.47,8.95
1,bge-base-en-v1.5,Hit@3,2.16,-1.23,5.40
2,bge-large-en-v1.5,Hit@1,11.42,8.02,14.97
3,bge-large-en-v1.5,Hit@3,9.10,5.71,12.50


In [7]:
izabran = MODELI[0]
najbolji_hit1 = np.mean(rang_small <= 1)
for ime in MODELI[1:]:
    red = razlike_df[(razlike_df["model"] == ime.split("/")[1]) & (razlike_df["mera"] == "Hit@1")]
    hit1 = np.mean(REZULTATI[ime]["rang_val"] <= 1)
    if red["p2.5"].iloc[0] > 0 and hit1 > najbolji_hit1:
        izabran = ime
        najbolji_hit1 = hit1

print("IZABRAN MODEL:", izabran)

IZABRAN MODEL: BAAI/bge-large-en-v1.5


In [8]:
tezine = {}
for ime in MODELI:
    r = REZULTATI[ime]
    kolona = {}
    for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
        S = (1 - w) * r["S_prozor"] + w * r["S_dokument"]
        kolona[w] = np.mean(rang_tacne(S, TACNA_VAL) <= 1) * 100
    tezine[ime.split("/")[1]] = kolona

tabela_tezine = pd.DataFrame(tezine)
tabela_tezine.index.name = "w"
print("Hit@1 (%) na val upitima:")
tabela_tezine.round(1)

Hit@1 (%) na val upitima:


,bge-small-en-v1.5,bge-base-en-v1.5,bge-large-en-v1.5
w,,,
0.0,39.8,44.4,51.4
0.1,41.5,48.5,52.9
0.2,42.9,49.1,54.5
0.3,43.8,49.5,55.2
0.4,42.7,48.0,55.7
0.5,40.9,45.8,54.2
0.7,34.4,40.3,49.5
1.0,27.6,34.1,40.4
